# Generate Postcode DC Scope

Builds a Data Correctness (DC) scoping CSV that sets `addr:postcode` on APTs, from a
MNR-vs-Orbis comparison CSV (columns `orbis_id`, `mnr_postal_code`, street / house number /
place match details, geometry WKT, distance, ...).

## What this notebook does

1. **Read the input CSV** with pandas (`orbis_id` and `mnr_postal_code` are read as text so
   long ids and leading-zero postcodes survive)
2. **Drop rows** that have no `orbis_id` or no `mnr_postal_code` — every other row is kept as-is
3. **Map columns** to the DC scope format and write the output CSV
4. **Validate** the output: row count, column order, constant values, no blanks in the required fields

## Column mapping

| Output column | Source |
|---|---|
| `Product_orbis_id` | `orbis_id` |
| `Target_Value` | `mnr_postal_code` |
| `Product_Address_component` | constant `addr:postcode` |
| `Product_Language` | constant `es-Latn` |
| `Source_Feature_Id` | blank |
| `Lat` | constant `0` |
| `Lon` | constant `0` |

### Load input CSV

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

INPUT_CSV = "/Users/sawan.darekar/Downloads/IM-74527-ARG/Additional_Postal_Scope_ARG.csv"  # <-- update this path

# keep both key columns as text: orbis_id is a long composite id, postcodes can have leading zeros
input_df = pd.read_csv(INPUT_CSV, dtype={"orbis_id": str, "mnr_postal_code": str})

print(f"Input rows    : {len(input_df)}")
print(f"Input columns : {list(input_df.columns)}")
display(input_df.head())

Input rows    : 1101872
Input columns : ['orbis_id', 'feat_id', 'mnr_postal_code', 'orbis_street', 'mnr_street', 'orbis_house_number', 'mnr_house_number', 'orbis_place', 'mnr_place', 'orbis_dependent_area', 'mnr_dependent_area', 'match_distance_m', 'attribute_score', 'match_quality', 'orbis_geometry_wkt', 'distance_m', 'mnr_geometry_wkt']


,orbis_id,feat_id,mnr_postal_code,orbis_street,mnr_street,orbis_house_number,mnr_house_number,orbis_place,mnr_place,orbis_dependent_area,mnr_dependent_area,match_distance_m,attribute_score,match_quality,orbis_geometry_wkt,distance_m,mnr_geometry_wkt
0,14533_3583805932224446464_11446580780074784333,000dcd0c-cc15-4cf9-8836-d23407dd5800,B1803FOR,Calle Doctor Cosme Argerich,Calle Doctor Cosme Argerich,890,890,Provincia de Buenos Aires,Provincia de Buenos Aires,La Unión,La Unión,NaN,11,street_house_place_area,POINT (-58.5261507 -34.8679528),1.30,POINT (-58.5261582 -34.8679428)
1,14533_3720430683554119680_619410115419435539,00129852-4d74-4419-8d67-7a16fa7fc6b9,A4406BDB,Avenida Uruguay,Avenida Uruguay,541,541,Salta,Salta,Salta,NaN,NaN,10,street_house_place,POINT (-65.4000826 -24.7853557),0.00,POINT (-65.4000826 -24.7853557)
2,14533_3680102734194606080_10513354397669441572,00135202-a1f6-461e-8102-46842ed9c6bf,S2152CAA,Calle Arenales,Calle Arenales,33,33,Santa Fe,Santa Fe,Las Malvinas,Rosario,NaN,10,street_house_place,POINT (-60.6622139 -32.9259067),0.00,POINT (-60.6622139 -32.9259067)
3,14533_3573058552004149248_14613972282096569959,002143e1-eb93-4ea7-a215-7a19a0299d6b,B7630DEF,Avenida 42,Avenida 42,2509,2509,Provincia de Buenos Aires,Provincia de Buenos Aires,Necochea,Necochea - Quequén,NaN,10,street_house_place,POINT (-58.7282339 -38.5581023),3.62,POINT (-58.7282038 -38.5581248)
4,14533_3680099139981737984_10335865846750548853,00218c5e-88f0-4af3-8510-05f8acfe612a,S2000FPD,Calle Juan Manuel de Rosas,Calle Juan Manuel de Rosas,1884,1884,Santa Fe,Santa Fe,República de la Sexta,Rosario,NaN,10,street_house_place,POINT (-60.6346033 -32.9606135),4.28,POINT (-60.6346482 -32.9606061)


### Generate DC scope CSV

In [3]:
OUTPUT_CSV = "/Users/sawan.darekar/Downloads/IM-74527-ARG/IM-60432_Additional_Postal_Scope_ARG.csv"  # <-- update this pathhttps://tomtom.atlassian.net/browse/
ADDRESS_COMPONENT = "addr:postcode"
LANGUAGE = "es-Latn"

SCOPE_COLUMNS = [
    "Product_orbis_id",
    "Target_Value",
    "Product_Address_component",
    "Product_Language",
    "Source_Feature_Id",
    "Lat",
    "Lon",
]

work_df = input_df[["orbis_id", "mnr_postal_code"]].copy()
work_df["orbis_id"] = work_df["orbis_id"].astype("string").str.strip()
work_df["mnr_postal_code"] = work_df["mnr_postal_code"].astype("string").str.strip()

# treat empty strings as missing, then drop the incomplete rows (no other filtering)
work_df = work_df.replace("", pd.NA)
dropped_df = work_df[work_df["orbis_id"].isna() | work_df["mnr_postal_code"].isna()]
work_df = work_df.drop(dropped_df.index)

scope_df = pd.DataFrame({
    "Product_orbis_id": work_df["orbis_id"].to_numpy(),
    "Target_Value": work_df["mnr_postal_code"].to_numpy(),
    "Product_Address_component": ADDRESS_COMPONENT,
    "Product_Language": LANGUAGE,
    "Source_Feature_Id": "",
    "Lat": 0,
    "Lon": 0,
})[SCOPE_COLUMNS]

scope_df.to_csv(OUTPUT_CSV, index=False)

In [4]:
print(f"Input rows                      : {len(input_df)}")
print(f"Dropped (missing id / postcode) : {len(dropped_df)}")
print(f"Scope rows written              : {len(scope_df)}")
print(f"Unique orbis ids                : {scope_df['Product_orbis_id'].nunique()}")
print(f"Output file                     : {OUTPUT_CSV}")
display(scope_df.head(10))

Input rows                      : 1101872
Dropped (missing id / postcode) : 0
Scope rows written              : 1101872
Unique orbis ids                : 1101872
Output file                     : /Users/sawan.darekar/Downloads/IM-74527-ARG/IM-60432_Additional_Postal_Scope_ARG.csv


,Product_orbis_id,Target_Value,Product_Address_component,Product_Language,Source_Feature_Id,Lat,Lon
0,14533_3583805932224446464_11446580780074784333,B1803FOR,addr:postcode,es-Latn,,0,0
1,14533_3720430683554119680_619410115419435539,A4406BDB,addr:postcode,es-Latn,,0,0
2,14533_3680102734194606080_10513354397669441572,S2152CAA,addr:postcode,es-Latn,,0,0
3,14533_3573058552004149248_14613972282096569959,B7630DEF,addr:postcode,es-Latn,,0,0
4,14533_3680099139981737984_10335865846750548853,S2000FPD,addr:postcode,es-Latn,,0,0
5,14533_3491199603584073728_9003335042965283281,R8403CBU,addr:postcode,es-Latn,,0,0
6,14533_3680104028444360704_12416860622340267085,S2013SOU,addr:postcode,es-Latn,,0,0
7,14533_3583961553615257600_14771672646618938717,B1873BYJ,addr:postcode,es-Latn,,0,0
8,14533_3583913004879118336_18231765980079446517,B1823BOW,addr:postcode,es-Latn,,0,0
9,14533_3491280761541689344_10028001002182723177,R8402DLC,addr:postcode,es-Latn,,0,0


### Validate output CSV

In [5]:
check_df = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)

checks = {
    "column order matches expected": list(check_df.columns) == SCOPE_COLUMNS,
    "row count matches input minus dropped": len(check_df) == len(input_df) - len(dropped_df),
    "no blank Product_orbis_id": (check_df["Product_orbis_id"].str.strip() != "").all(),
    "no blank Target_Value": (check_df["Target_Value"].str.strip() != "").all(),
    "Product_Address_component is constant": set(check_df["Product_Address_component"]) == {ADDRESS_COMPONENT},
    "Product_Language is constant": set(check_df["Product_Language"]) == {LANGUAGE},
    "Source_Feature_Id is blank": (check_df["Source_Feature_Id"] == "").all(),
    "Lat is 0": (check_df["Lat"] == "0").all(),
    "Lon is 0": (check_df["Lon"] == "0").all(),
}

for name, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

print()
print("All checks passed." if all(checks.values()) else "WARNING: some checks failed!")

  PASS  column order matches expected
  PASS  row count matches input minus dropped
  PASS  no blank Product_orbis_id
  PASS  no blank Target_Value
  PASS  Product_Address_component is constant
  PASS  Product_Language is constant
  PASS  Source_Feature_Id is blank
  PASS  Lat is 0
  PASS  Lon is 0

All checks passed.
